<a href="https://colab.research.google.com/github/Yuliia-Safonova/Analysis_Cafe_Sales/blob/main/EDA_Cafe_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analysis Cafe Sales

*Cafe Sales - Dirty Data for Cleaning Training*

**Overview**  

The Dirty Cafe Sales dataset contains 10,000 rows of synthetic data representing sales transactions in a cafe. This dataset is intentionally "dirty," with missing values, inconsistent data, and errors introduced to provide a realistic scenario for data cleaning and exploratory data analysis (EDA). It can be used to practice cleaning techniques, data wrangling, and feature engineering.

**File Information**  
- File Name: dirty_cafe_sales.csv  
- Number of Rows: 10,000  
- Number of Columns: 8

**Columns Description**  

- `Transaction ID` - A unique identifier for each transaction. Always present and unique.  

- `Item` -	The name of the item purchased. May contain missing or invalid values (e.g., "ERROR").  

- `Quantity` -	The quantity of the item purchased. May contain missing or invalid values.

- `Price Per Unit` -	The price of a single unit of the item. May contain missing or invalid values.

- `Total Spent` -	The total amount spent on the transaction. Calculated as Quantity * Price Per Unit.

- `Payment Method` -	The method of payment used. May contain missing or invalid values (e.g., None, "UNKNOWN").

- `Location` -	The location where the transaction occurred. May contain missing or invalid values.

- `Transaction Date` -	The date of the transaction. May contain missing or incorrect values.

` !pip install numpy pandas matplotlib `

In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Loading dataset

In [46]:
url = 'https://raw.githubusercontent.com/Yuliia-Safonova/Analysis_Cafe_Sales/refs/heads/main/data/dirty_cafe_sales.csv'

df_origin = pd.read_csv(url)
df_origin.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [47]:
df = df_origin.copy()

## Primary diagnosis

In [48]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [50]:
print("Data type (raw) :\n")
df.dtypes

Data type (raw) :



,0
Transaction ID,object
Item,object
Quantity,object
Price Per Unit,object
Total Spent,object
Payment Method,object
Location,object
Transaction Date,object


Problem:  
- Numeric columns (Quantity, Price Per Unit, Total Spent)
- Date column (Transaction Date)

In [51]:
print('Missing values:\n')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_pct
missing_report = pd.DataFrame({
    'count': missing,
    '%': missing_pct
})
missing_report[missing_report['count'] > 0]

Missing values:



,count,%
Item,333,3.3
Quantity,138,1.4
Price Per Unit,179,1.8
Total Spent,173,1.7
Payment Method,2579,25.8
Location,3265,32.6
Transaction Date,159,1.6


In [52]:
print("Unique values:\n")

print("---Item---\n")
display(df['Item'].value_counts(dropna=False))

print("\n---Payment Method---\n")
display(df['Payment Method'].value_counts(dropna=False))

print("\n---Location---\n")
display(df['Location'].value_counts(dropna=False))

Unique values:

---Item---



,count
Item,
Juice,1171
Coffee,1165
Salad,1148
Cake,1139
Sandwich,1131
Smoothie,1096
Cookie,1092
Tea,1089
UNKNOWN,344



---Payment Method---



,count
Payment Method,
NaN,2579
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293



---Location---



,count
Location,
NaN,3265
Takeaway,3022
In-store,3017
ERROR,358
UNKNOWN,338


In [53]:
print("Find lines with 'ERROR' or 'UNKNOWN'")
display(df.isin(['ERROR', 'UNKNOWN']).any())

mask_error = df.isin(['ERROR', 'UNKNOWN']).any(axis=1)
print(f"\nLines with 'ERROR' or 'UNKNOWN': {mask_error.sum()} ")

Find lines with 'ERROR' or 'UNKNOWN'


,0
Transaction ID,False
Item,True
Quantity,True
Price Per Unit,True
Total Spent,True
Payment Method,True
Location,True
Transaction Date,True



Lines with 'ERROR' or 'UNKNOWN': 2845 


Additionally: Instead of True/False, record the number of ‘ERROR’ or ‘UNKNOWN’ entries in each column

In [54]:
print("Find lines with 'ERROR' or 'UNKNOWN'")
display(df.isin(['ERROR', 'UNKNOWN']).sum())

Find lines with 'ERROR' or 'UNKNOWN'


,0
Transaction ID,0
Item,636
Quantity,341
Price Per Unit,354
Total Spent,329
Payment Method,599
Location,696
Transaction Date,301


## Data cleaning

If a copy of the dataset has not been made, you need to make one here

- ` df_origin ` - origin dataset  
- ` df ` - copy origin dataset

In [55]:
print('Rename columns:\n')

df.columns = df.columns.str.strip()
# df.columns.str.lower().str.replace(" ", "_")
df = df.rename(columns={
    'Transaction ID': 'transaction_id',
    'Item': 'item',
    'Quantity': 'quantity',
    'Price Per Unit': 'price_per_unit',
    'Total Spent': "total_spent",
    'Payment Method': 'payment_method',
    'Location': 'location',
    'Transaction Date': 'transaction_date'
})

print(list(df.columns))

Rename columns:

['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent', 'payment_method', 'location', 'transaction_date']


In [56]:
# 'ERROR' / 'UNKNOW' -> NaN
df.replace(['ERROR', 'UNKNOWN', 'error', 'unknow', ''], np.nan, inplace=True)
df

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [57]:
# Converting numeric columns
for col in ['quantity', 'price_per_unit', 'total_spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Date conversion
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

In [58]:
print("Converting columns:\n")
df.dtypes

Converting columns:



,0
transaction_id,object
item,object
quantity,float64
price_per_unit,float64
total_spent,float64
payment_method,object
location,object
transaction_date,datetime64[ns]


In [59]:
# Extracting time signatures
df['year'] = df['transaction_date'].dt.year.astype('Int64')
df['month'] = df['transaction_date'].dt.month.astype('Int8')

# df['month_name'] = df['transaction_date'].dt.month_name()
df['month_name'] = df['transaction_date'].dt.strftime('%B')

df['weekday'] = df['transaction_date'].dt.day_name()
df['week'] = df['transaction_date'].dt.isocalendar().week.astype('Int8')

# display(df[['transaction_date', 'year', 'month', 'month_name', 'weekday', 'week']].dtypes)
df[['transaction_date', 'year', 'month', 'month_name', 'weekday', 'week']].head()

,transaction_date,year,month,month_name,weekday,week
0,2023-09-08,2023,9,September,Friday,36
1,2023-05-16,2023,5,May,Tuesday,20
2,2023-07-19,2023,7,July,Wednesday,29
3,2023-04-27,2023,4,April,Thursday,17
4,2023-06-11,2023,6,June,Sunday,23


In [60]:
# Restoring total_spent (quantity * price)

mask_recoverable = (
    df['total_spent'].isna() &
    df['quantity'].notna() &
    df['price_per_unit'].notna()
)

df.loc[mask_recoverable, 'total_spent'] = df.loc[mask_recoverable, 'quantity'] * df.loc[mask_recoverable, 'price_per_unit']

print(f"Restoring total_spent: {mask_recoverable.sum()} string(quantity * price)")

Restoring total_spent: 462 string(quantity * price)


In [61]:
# dict of median price
price_map = (
    df.dropna(subset=['item', 'price_per_unit'])
    .groupby('item')['price_per_unit']
    .median()
    .to_dict()
)

mask_price = df['price_per_unit'].isna() & df['item'].notna()
df.loc[mask_price, 'price_per_unit'] = df.loc[mask_price, 'item'].map(price_map)

print(f"Restoring price_per_unit: {mask_price.sum()} string(median per item)")

Restoring price_per_unit: 479 string(median per item)


In [62]:
# Restoring total_spent (new price - median item)

mask_recoverable2 = (
    df['total_spent'].isna() &
    df['quantity'].notna() &
    df['price_per_unit'].notna()
)

df.loc[mask_recoverable2, 'total_spent'] = df.loc[mask_recoverable2, 'quantity'] * df.loc[mask_recoverable2, 'price_per_unit']

print(f"Restoring total_spent: {mask_recoverable2.sum()} string(quantity * price)")

Restoring total_spent: 17 string(quantity * price)


In [63]:
# fillna item -> mode
mode_item = df['item'].mode()[0]

# df['item'].fillna(mode_item, inplace=True) # old version
df.fillna({'item': mode_item}, inplace=True)

In [64]:
# payment_method, location -> 'Unknow'
df.fillna(
    {
        'payment_method': 'Unknow',
        'location': 'Unknow'
    },
    inplace=True
)

In [65]:
# Delete transaction_date where NaT or NaN
befor = len(df)

df.dropna(subset=['transaction_date', 'quantity'], inplace=True)

after = len(df)

print(f"Deleted {befor - after} rows")

Deleted 914 rows


In [66]:
# Delete duplicates
df.duplicated().sum()   # 0
df.drop_duplicates(inplace=True)

In [67]:
# Checking data for logic
(df.quantity <= 0).sum()        # 0
(df.price_per_unit <= 0).sum()  # 0
(df.total_spent  <= 0).sum()    # 0

np.int64(0)

In [68]:
(df['quantity'].isna() & df['price_per_unit'].isna()).sum()

np.int64(0)

In [69]:
(df['quantity'].isna() | df['price_per_unit'].isna()).sum()

np.int64(48)

In [70]:
# Restoring price_per_unit (total_spent / quantity)

mask_recoverable3 = (
    df['total_spent'].notna() &
    df['quantity'].notna() &
    df['price_per_unit'].isna()
)

df.loc[mask_recoverable3, 'price_per_unit'] = df.loc[mask_recoverable3, 'total_spent'] / df.loc[mask_recoverable3, 'quantity']

print(f"Restoring price_per_unit: {mask_recoverable3.sum()} string(total_spent / quantity)")

Restoring price_per_unit: 45 string(total_spent / quantity)


In [71]:
# Delete rows with NaN
df.dropna(subset=['total_spent'], inplace=True)

print(f"Delete rows with NaN (df['total_spent'].isna() & df['price_per_unit'].isna())")

Delete rows with NaN (df['total_spent'].isna() & df['price_per_unit'].isna())


In [72]:
# Check quantity * price_per_unit = total_spent
mask_total_spent = (df.quantity * df.price_per_unit != df.total_spent)
print(f"Uncorrect rows total_spent: {mask_total_spent.sum()}")

Uncorrect rows total_spent: 0


In [73]:
# Text fields are standartized (Title Case)
for col in ['item', 'payment_method', 'location']:
    df[col] = df[col].str.strip().str.title()

print("Text fields are standartized (Title Case)")

Text fields are standartized (Title Case)


## Cleaning results

In [81]:
print("Cleaning results:\n")
print("=" * 30, "\n")

print(f"Rows before: {len(df_origin)}")
print(f"Rows after: {len(df)}")
print(f"Loss: {len(df_origin) - len(df)} ({(len(df_origin) - len(df)) / len(df_origin) * 100:.1f}%)")

print("\n--- Residual gaps ---")
remaining = df.isna().sum()
print(remaining[remaining > 0] if remaining.any() else "No gaps")

print("\n--- Data types after cleanup: ---")
display(df.dtypes)

print("\n--- Statistics ---\n")
print(df[['quantity', 'price_per_unit', 'total_spent']].describe().round(2))

Cleaning results:


Rows before: 10000
Rows after: 9083
Loss: 917 (9.2%)

--- Residual gaps ---
No gaps

--- Data types after cleanup: ---


,0
transaction_id,object
item,object
quantity,float64
price_per_unit,float64
total_spent,float64
payment_method,object
location,object
transaction_date,datetime64[ns]
year,Int64
month,Int8



--- Statistics ---

       quantity  price_per_unit  total_spent
count   9083.00         9083.00      9083.00
mean       3.03            2.95         8.93
std        1.42            1.28         6.00
min        1.00            1.00         1.00
25%        2.00            2.00         4.00
50%        3.00            3.00         8.00
75%        4.00            4.00        12.00
max        5.00            5.00        25.00
